In [ ]:
# =========================================================
# CONFIGURAÇÃO INICIAL DO NOTEBOOK
# =========================================================

%reload_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd

sys.path.append(os.path.abspath(".."))

from src.tratamento_dados import TratamentoDados

In [ ]:
# =========================================================
# CARREGANDO O DATASET
# =========================================================

caminho_arquivo = "../data/desafio_nps_fase_1.csv"

df = pd.read_csv(caminho_arquivo)

df.head()

In [ ]:
# =========================================================
# TRATAMENTO INICIAL DOS DADOS
# =========================================================

# Instancia a classe responsável pelo tratamento da base
tratamento = TratamentoDados(df)

# Exibe diagnóstico inicial:
# quantidade de linhas, colunas, duplicados e valores nulos
tratamento.diagnostico_inicial()

# Remove registros duplicados da base
tratamento.remover_duplicados()

# Trata valores nulos:
# - colunas numéricas → mediana
# - colunas categóricas → moda
tratamento.tratar_nulos_simples()

# Remove colunas identificadoras que não devem participar
# do treinamento do modelo preditivo, pois representam apenas
# IDs de controle e não variáveis explicativas do problema
tratamento.remover_colunas([
    "customer_id",
    "order_id"
])

# Cria a variável alvo categórica para classificação NPS:
# 0-6 → Detrator
# 7-8 → Neutro
# 9-10 → Promotor
tratamento.criar_classe_nps()

# Cria a classificação de satisfação do cliente (CSAT):
# 0-2   → Muito insatisfeito
# 2-4   → Insatisfeito
# 4-6   → Indiferente
# 6-8   → Satisfeito
# 8-10  → Muito satisfeito
tratamento.criar_classe_csat()

# Retorna o dataframe final após todas as transformações
df_tratado = tratamento.obter_dataframe()

# Visualiza as primeiras linhas da base tratada
df_tratado.head()

In [ ]:
# =========================================================
# PREPARAÇÃO DA BASE PARA MODELAGEM
# =========================================================

# Define quais colunas não devem participar do modelo
# pois podem causar vazamento de informação (data leakage)
# ou representam variáveis derivadas da própria variável alvo
colunas_remover_modelo = [
    "nps_score",
    "csat_internal_score",
    "classe_csat"
]

# Prepara a base:
# X = variáveis explicativas
# y = variável alvo (classe_nps)
X, y = tratamento.preparar_base_modelagem(
    coluna_alvo="classe_nps",
    colunas_remover=colunas_remover_modelo,
    drop_first=True,
    mostrar_saida=True
)

# Visualiza as primeiras linhas da base pronta para modelagem
print(X.head())

# Visualiza a variável alvo
print(y.head())

In [ ]:
# =========================================================
# PREPARAÇÃO DA BASE PARA MODELAGEM
# =========================================================

# Define quais colunas não devem participar do modelo
# pois podem causar vazamento de informação (data leakage)
# ou representam variáveis derivadas da própria variável alvo
colunas_remover_modelo = [
    "nps_score",
    "csat_internal_score",
    "classe_csat"
]

# Prepara a base:
# X = variáveis explicativas
# y = variável alvo (classe_nps)
X, y = tratamento.preparar_base_modelagem(
    coluna_alvo="classe_nps",
    colunas_remover=colunas_remover_modelo,
    drop_first=True,
    mostrar_saida=True
)

# Visualiza as primeiras linhas da base pronta para modelagem
print(X.head())

# Visualiza a variável alvo
print(y.head())

In [ ]:
# =========================================================
# MODELAGEM PREDITIVA RANDOM FOREST
# O Random Forest permite antecipar o comportamento do cliente, transformando sinais operacionais em decisões preventivas de experiência.
# =========================================================

from src.analise_preditiva import AnalisePreditiva

# Instancia a classe de análise preditiva usando a base preparada
preditiva = AnalisePreditiva(X, y)

# Divide a base em treino e teste
preditiva.dividir_treino_teste(
    test_size=0.2,
    random_state=42,
    stratify=True
)

# Treina o primeiro modelo: Random Forest
modelo_rf = preditiva.treinar_random_forest()


# =========================================================
# VISUALIZANDO AS PREVISÕES DO MODELO
# =========================================================

# Retorna os resultados consolidados
resultados_modelos = preditiva.obter_resultados()

resultados_modelos


In [ ]:
""" analise_previsoes = preditiva.analisar_previsoes(
    mostrar_saida=True,
    qtd_exibir=20
)

analise_previsoes["comparacao_previsoes"].head(20)

analise_previsoes["matriz_confusao"]

analise_previsoes["relatorio_classificacao"]

analise_previsoes["accuracy_manual"]

preditiva.exportar_matriz_confusao(
    salvar_em="../images/matriz_confusao_random_forest.png",
    mostrar_saida=True
) """

In [ ]:
# ============================================================
# TREINAMENTO DOS 4 MODELOS RESTANTES + COMPARAÇÃO DOS 6
# ============================================================

# 3. SVM
preditiva.treinar_svm()
preditiva.analisar_previsoes()


# 4. SVM + StandardScaler
preditiva.treinar_svm_com_scaler()
preditiva.analisar_previsoes()


# 5. Extra Trees
preditiva.treinar_extra_trees()
preditiva.analisar_previsoes()


# 6. Árvore de Decisão
preditiva.treinar_arvore_decisao()
preditiva.analisar_previsoes()



# ============================================================
# COMPARAÇÃO FINAL DOS 6 MODELOS
# ============================================================

comparacao_modelos = preditiva.obter_resultados()

comparacao_modelos = comparacao_modelos.sort_values(
    by="f1_macro",
    ascending=False
)

print("=" * 70)
print("COMPARAÇÃO FINAL DOS MODELOS")
print("=" * 70)

display(comparacao_modelos)